# Prepare the global WAC texture

This notebook downloads the LROC WAC global 16 ppd GeoTIFF, downsamples it for the browser, stretches its grayscale values, and downloads `wac-texture.jpg`. Save that image in the repo at `public/data/wac-texture.jpg`.

In [ ]:
%pip -q install rasterio pillow requests

In [ ]:
from pathlib import Path
import json
import requests
import numpy as np
import rasterio
from rasterio.enums import Resampling
from PIL import Image

source_url = "https://pds.lroc.im-ldi.com/data/LRO-L-LROC-5-RDR-V1.0/LROLRC_2001/EXTRAS/BROWSE/WAC_GLOBAL/WAC_GLOBAL_E000N0000_016P.TIF"
source_path = Path("WAC_GLOBAL_E000N0000_016P.TIF")
output_path = Path("wac-texture.jpg")
metadata_path = Path("wac-texture.json")

if not source_path.exists():
    response = requests.get(source_url, stream=True, timeout=(20, 180))
    response.raise_for_status()
    with source_path.open("wb") as output:
        for chunk in response.iter_content(chunk_size=1024 * 1024):
            if chunk:
                output.write(chunk)

print(f"Downloaded {source_path.stat().st_size / 1_000_000:.1f} MB")

In [ ]:
output_width = 2048
output_height = 1024

with rasterio.open(source_path) as dataset:
    band = dataset.read(
        1,
        out_shape=(output_height, output_width),
        masked=True,
        resampling=Resampling.average,
    )
    values = np.asarray(band.filled(np.nan), dtype=np.float32)

valid = values[np.isfinite(values)]
if valid.size == 0:
    raise ValueError("The WAC raster has no valid pixels.")

low, high = np.percentile(valid, [1, 99.5])
if high <= low:
    raise ValueError("Could not calculate a useful contrast stretch.")

scaled = np.clip((values - low) / (high - low), 0, 1)
scaled = np.nan_to_num(scaled, nan=0.0)
grayscale = (scaled * 255).round().astype(np.uint8)
Image.fromarray(grayscale, mode="L").convert("RGB").save(
    output_path,
    format="JPEG",
    quality=90,
    optimize=True,
)

metadata = {
    "source": source_url,
    "product": "LROC WAC Global Morphologic Mosaic, E000N0000, 16 ppd",
    "output": output_path.name,
    "width": output_width,
    "height": output_height,
    "stretchPercentiles": [1, 99.5],
    "stretchValues": [float(low), float(high)],
    "resampling": "average",
    "note": "Grayscale 643 nm morphology texture, not natural-color imagery.",
}
metadata_path.write_text(json.dumps(metadata, indent=2) + "\n")
print(f"Created {output_path} ({output_path.stat().st_size / 1_000_000:.1f} MB)")
print(metadata_path.read_text())
Image.open(output_path).resize((1024, 512)).show()

In [ ]:
from google.colab import files
files.download(str(output_path))